In [1]:
"""
Setup for propagating through turbulent atmosphere. This consists of defining the propagation geometry and turbulent properties.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, Bounds

def fun(X):
    return 
#### Determine Geometry ####
D2 = 0.5            # Diameter of the observation plane [m]
wvl = 1e-6          # Wavelength [m]
k = 2*np.pi/wvl     # Wavenumber [rad/m]
Dz = 50e3           # Propagation distance [m]

#### Point Source Properties ####
DROI = 4*D2         # Diameter of region of interest [m]
D1 = wvl*Dz/DROI    # Diameter of central lobe [m]
R = Dz              # Wavefront radius of curvature [m]

#### Atmospheric Properties ####
Cn2 = 1e-16         # Refractive index structure parameter [m^-2/3]
r0sw = (0.423*k**2*Cn2*3/8*Dz)**(-3/5)  # Spherical wave coherence diameter [m]
r0pw = (0.423*k**2*Cn2*Dz)**(-3/5)      # Plane wave coherence diameter [m]
p = np.linspace(0,Dz,int(1e3))

#### Log-Amplitude Variance ####
rytov = 0.563*k**(7/6)*np.sum(Cn2*(1-p/Dz)**(5/6)\
                              *p**(5/6)*(p[1]-p[0]))    # Equation 9.69 (J.Schmidt)

#### Screen Properties ####
nscr = 11
A = np.zeros((2,nscr))
alpha = np.arange(0,nscr)/(nscr-1)
A[0,:] = alpha**(5/3)
A[1,:] = (1-alpha)**(5/6) * alpha**(5/6)
b = np.array([[r0sw**(-5/3)],[rytov/1.33*(k/Dz)**(5/6)]])

#### Initial Guess ####
x0 = (nscr/3 * r0sw * np.ones((nscr,1)))**(-5/3)
fun = lambda X: np.sum((A@X.reshape(-1,1,order='F')-b)**2)
x1 = np.zeros((nscr,1))
rmax = 0.1
x2 = rmax/1.33*(k/Dz)**(5/6)/A[1,:]
x2[A[1,:] == 0] = 50**(-5/3)
bounds = Bounds(x1.flatten(),x2)
min_results = minimize(fun,x0.flatten(),method='trust-constr',bounds=bounds)
r0scrn = min_results.x**(-3/5)
r0scrn[np.isinf(r0scrn)] = 1e6
print(min_results)

           message: `gtol` termination condition is satisfied.
           success: True
            status: 1
               fun: 4.466913358396056e-15
                 x: [ 7.368e-04  5.182e+00  6.282e+00  7.248e+00  8.078e+00
                      8.767e+00  9.361e+00  9.892e+00  1.023e+01  1.046e+01
                      7.368e-04]
               nit: 36
              nfev: 312
              njev: 26
              nhev: 0
          cg_niter: 74
      cg_stop_cond: 4
              grad: [ 0.000e+00  3.949e-09  4.626e-09  3.464e-09  1.020e-09
                     -1.981e-09 -4.269e-09 -4.016e-09  6.286e-11  1.053e-08
                     -1.129e-07]
   lagrangian_grad: [ 3.961e-18  2.276e-09  3.660e-09  3.184e-09  1.456e-09
                     -8.873e-10 -2.914e-09 -3.095e-09  1.702e-10  9.928e-09
                     -3.064e-14]
            constr: [array([ 7.368e-04,  5.182e+00,  6.282e+00,  7.248e+00,
                            8.078e+00,  8.767e+00,  9.361e+00,  9.892e+00,
     

/tmp/ipykernel_8393/1670584969.py:45: RuntimeWarning: divide by zero encountered in divide
  x2 = rmax/1.33*(k/Dz)**(5/6)/A[1,:]
